# Evaluation of encoders (Within-subject retrieval)

Our encoder

In [7]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from fmri_mapping.embedding.evaluation import accuracy_cosine_similarity
from fmri_mapping.embedding.ops import split_repetitions
from fmri_mapping.io.nsd import get_resource, get_image
import torch
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F

from PIL import Image
from skimage.metrics import structural_similarity as ssim
from tqdm.notebook import tqdm, trange

from skimage.metrics import structural_similarity as ssim


In [28]:
def compute_ssim(image_source_ids, image_matched_ids):
    """
    Mean SSIM between source and matched images.

    - Uses get_image(id) -> PIL.Image
    - If ids match: imputes SSIM = 1.0 (no image loading)

    Returns:
      mean SSIM (float)
    """
    if len(image_source_ids) != len(image_matched_ids):
        raise ValueError("Inputs must have the same length.")

    n = len(image_source_ids)
    if n == 0:
        return float("nan")

    total_ssim = 0.0

    for src_id, tgt_id in zip(image_source_ids, image_matched_ids):
        if src_id == tgt_id:
            total_ssim += 1.0
            continue

        img1 = get_image(src_id).convert("L")
        img2 = get_image(tgt_id).convert("L")

        x = np.array(img1)
        y = np.array(img2)

        if x.shape != y.shape:
            raise ValueError("Images must have the same shape for SSIM.")

        total_ssim += float(ssim(x, y, data_range=255))

    return total_ssim / n


def initialize_clip_model(
    model_name: str = "openai/clip-vit-base-patch32",
    device: str = "cuda",
):
    model = CLIPModel.from_pretrained(model_name).to(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    model.eval()
    return model, processor


def compute_mean_clip_distance(
    image_source_ids,
    image_matched_ids,
    model,
    processor,
    device: str = "cuda",
    batch_size: int = 32,
) -> float:
    """
    Mean CLIP cosine similarity.

    - If ids match: contributes 1.0 without computing embeddings
    - Assumes get_image(id) exists
    """
    if len(image_source_ids) != len(image_matched_ids):
        raise ValueError("Inputs must have same length.")

    n = len(image_source_ids)
    if n == 0:
        return float("nan")

    same_count = 0
    pairs = []

    for src_id, tgt_id in zip(image_source_ids, image_matched_ids):
        if src_id == tgt_id:
            same_count += 1
        else:
            pairs.append((src_id, tgt_id))

    if not pairs:
        return 1.0

    sim_sum = float(same_count)

    with torch.no_grad():
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]

            source_images = [get_image(s).convert("RGB") for s, _ in batch]
            matched_images = [get_image(t).convert("RGB") for _, t in batch]

            src_inputs = processor(images=source_images, return_tensors="pt", padding=True).to(device)
            tgt_inputs = processor(images=matched_images, return_tensors="pt", padding=True).to(device)

            src_feat = F.normalize(model.get_image_features(**src_inputs), dim=-1)
            tgt_feat = F.normalize(model.get_image_features(**tgt_inputs), dim=-1)

            sim_sum += (src_feat * tgt_feat).sum(dim=-1).sum().item()

    return sim_sum / n

In [32]:
def cka_linear_unbiased(X: torch.Tensor, Y: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    Unbiased linear CKA between two representation matrices X and Y.

    Assumes:
      - X: [n, dx]
      - Y: [n, dy]
      - same n, correct shapes

    Returns:
      scalar tensor
    """
    K = X @ X.T
    L = Y @ Y.T

    n = K.shape[0]

    K = K.clone()
    L = L.clone()
    K.fill_diagonal_(0)
    L.fill_diagonal_(0)

    def hsic_u(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        trace_term = (A * B).sum()
        sum_A = A.sum()
        sum_B = B.sum()
        ones_AB_ones = (A.sum(dim=1) * B.sum(dim=1)).sum()

        return (
            trace_term
            + (sum_A * sum_B) / ((n - 1) * (n - 2))
            - 2.0 * ones_AB_ones / (n - 2)
        ) / (n * (n - 3))

    hsic_xy = hsic_u(K, L)
    hsic_xx = hsic_u(K, K)
    hsic_yy = hsic_u(L, L)

    return hsic_xy / torch.sqrt(torch.clamp(hsic_xx * hsic_yy, min=eps))

def get_repetition_embeddings(rep_i: int, rep_j: int, other_rep: int, Z: np.ndarray, df_repetitions: pd.DataFrame, expected_samples: int = 515) -> tuple[np.ndarray, np.ndarray]:
    indexes_i = df_repetitions[f"subject_index_{rep_i}"]
    indexes_j = df_repetitions[f"subject_index_{rep_j}"]
    # Assert that there is not any entry nan in both at the same time
    assert not ((indexes_i.isna()) & (indexes_j.isna())).any()
    # Input the missings with the other repetition
    indexes_i = indexes_i.fillna(df_repetitions[f"subject_index_{other_rep}"]).values
    indexes_j = indexes_j.fillna(df_repetitions[f"subject_index_{other_rep}"]).values
    # Check not any nan
    assert not np.isnan(indexes_i).any()
    assert not np.isnan(indexes_j).any()

    Z_i = Z[indexes_i.astype(int)]
    Z_j = Z[indexes_j.astype(int)]

    nsd_ids = df_repetitions.nsd_id.values.astype(int)

    # assert 766 len
    assert Z_i.shape[0] == Z_j.shape[0] == len(nsd_ids) == expected_samples


    return Z_i, Z_j, nsd_ids

def compute_rsa(Z_i: torch.tensor, Z_j: torch.tensor, first_metric: str = "pearson", second_metric: str = "pearson") -> dict:

    # Compute RDMs
    if first_metric == "pearson":
        rdm_i = 1 - torch.corrcoef(Z_i)
        rdm_j = 1 - torch.corrcoef(Z_j)
    elif first_metric == "euclidean":
        rdm_i = torch.cdist(Z_i, Z_i)
        rdm_j = torch.cdist(Z_j, Z_j)
    elif first_metric == "cosine":
        rdm_i = 1 - torch.nn.functional.cosine_similarity(Z_i.unsqueeze(1), Z_i.unsqueeze(0), dim=-1)
        rdm_j = 1 - torch.nn.functional.cosine_similarity(Z_j.unsqueeze(1), Z_j.unsqueeze(0), dim=-1)

    # Obtain the upper triangular part of the RDMs

    rows, cols = torch.triu_indices(rdm_i.shape[0], rdm_i.shape[1], offset=1, device=rdm_i.device)
    rdm_i_upper = rdm_i[rows, cols]
    rdm_j_upper = rdm_j[rows, cols]

    
    # Compute the correlation between the upper triangular parts of the RDMs
    if second_metric == "pearson":
        rsa = torch.corrcoef(torch.stack([rdm_i_upper, rdm_j_upper]))[0, 1].item()
    elif second_metric == "spearman":
        rdm_i_upper = rdm_i_upper.cpu().numpy()
        rdm_j_upper = rdm_j_upper.cpu().numpy()
        rsa = pd.Series(rdm_i_upper).corr(pd.Series(rdm_j_upper), method="spearman")
    elif second_metric == "euclidean":
        rsa = torch.norm(rdm_i_upper - rdm_j_upper).item()
    elif second_metric == "cosine":
        rsa = torch.nn.functional.cosine_similarity(rdm_i_upper, rdm_j_upper, dim=0).item()

    return rsa

    
def compute_views_metrics(Z_i: torch.tensor, Z_j: torch.tensor, nsd_ids: np.ndarray, **kwargs) -> dict:
    metrics = dict(**kwargs)
    recall, cosine, mean_rank = accuracy_cosine_similarity(Z_i, Z_j)
    metrics["recall"] = recall
    metrics["cosine"] = cosine
    metrics["mean_rank"] = mean_rank
    metrics["rsa_pearson_pearson"] = compute_rsa(Z_i, Z_j, first_metric="pearson", second_metric="pearson")
    metrics["rsa_euclidean_pearson"] = compute_rsa(Z_i, Z_j, first_metric="euclidean", second_metric="pearson")
    metrics["unbiased_cka"] = cka_linear_unbiased(Z_i, Z_j).item()

    return metrics

def get_close_nsd_id(nsd_ids: np.ndarray, Z_i: torch.Tensor, Z_j: torch.Tensor) -> np.ndarray:
    # Compute the cosine similarity between the two views
    cosine_sim = torch.nn.functional.cosine_similarity(Z_i.unsqueeze(1), Z_j.unsqueeze(0), dim=-1)
    # Get the index of the closest nsd_id in Z_j for each nsd_id in Z_i
    closest_indices = torch.argmax(cosine_sim, dim=1).cpu().numpy()
    closest_nsd_ids = nsd_ids[closest_indices]
    return closest_nsd_ids


In [30]:
df_stimuli = get_resource("stimulus")

min_repetitions = 3
stimuli_list = df_stimuli.query("repetition == @min_repetitions -1 and shared and exists").groupby("nsd_id").size().reset_index().rename(columns={0: "n_subjects"}).query("n_subjects == 8").nsd_id.tolist()
len(stimuli_list)

515

In [33]:
all_results = []
embeddings_folder = Path("../scripts/mlp_embeddings")
for subject in trange(1, 9):
    data = torch.load(embeddings_folder / f"ws_mlp_v1_128_768_sub-{subject:02d}.pt")
    Z = data["Z"]
    df_repetitions = split_repetitions(subject=subject, shuffle_indexes=True, min_exists=1, random_state=42)
    df_repetitions = df_repetitions.query("nsd_id in @stimuli_list and shared")
    for rep_i, rep_j, inpute_rep in tqdm([(1, 2, 3), (1, 3, 2), (2, 3, 1), (2, 1, 3), (3, 1, 2), (3, 2, 1)], leave=False):
        Z_i, Z_j, nsd_ids = get_repetition_embeddings(
            rep_i=rep_i, rep_j=rep_j, other_rep=inpute_rep, Z=Z, df_repetitions=df_repetitions
        )
        results = compute_views_metrics(Z_i.cuda(), Z_j.cuda(), nsd_ids, rep_i=rep_i, rep_j=rep_j, subject=subject)
        
        # Get image based metrics
        close_ids = get_close_nsd_id(nsd_ids, Z_i.cuda(), Z_j.cuda())
        ssim_score = compute_ssim(close_ids, nsd_ids)
        results["ssim"] = ssim_score
        
        all_results.append(results)

df_results = pd.DataFrame(all_results)
df_results.to_parquet("within_subject_encoder_results-3-reps.parquet", index=False)
df_results

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

,rep_i,rep_j,subject,recall,cosine,mean_rank,rsa_pearson_pearson,rsa_euclidean_pearson,unbiased_cka,ssim
0,1,2,1,0.990291,0.695732,1.034951,0.594809,0.390060,0.432062,0.988073
1,1,3,1,0.988350,0.698027,1.052427,0.605864,0.425379,0.433144,0.989763
2,2,3,1,0.994175,0.699460,1.021359,0.600066,0.436295,0.438491,0.990065
3,2,1,1,0.986408,0.695732,1.031068,0.594809,0.390060,0.432062,0.992151
4,3,1,1,0.988350,0.698027,1.036893,0.605864,0.425379,0.433144,0.989841
5,3,2,1,0.988350,0.699460,1.017476,0.600066,0.436295,0.438491,0.995023
6,1,2,2,0.984466,0.706682,1.168932,0.613718,0.481258,0.485634,0.981604
7,1,3,2,0.986408,0.705821,1.145631,0.611556,0.511079,0.477050,0.990725
8,2,3,2,0.982524,0.704748,1.079612,0.608377,0.438247,0.468037,0.986454
9,2,1,2,0.976699,0.706682,1.180583,0.613718,0.481258,0.485634,0.987496


In [34]:
from dmf.alerts import send_message
send_message("within", attachment="within_subject_encoder_results-3-reps.parquet")

In [35]:
# Table 1 - 
df_results = pd.read_parquet("within_subject_encoder_results-3-reps.parquet")

df_results_1 = df_results.groupby("subject").aggregate({"mean_rank": "mean", "rsa_pearson_pearson": "mean", "recall": "mean", "cosine": "mean",
                                                        "unbiased_cka": "mean", "ssim": "mean"}).reset_index()
df_results_avg = df_results_1.mean()
df_results_avg.subject = -1
# Append as last row
df_results_1 = pd.concat([df_results_1, pd.DataFrame([{"subject": "average", **df_results_avg}])], ignore_index=True)
df_results_1.subject = df_results_1.subject.astype(int).astype(str)
df_results_1.mean_rank = df_results_1.mean_rank.round(2).astype(str)
df_results_1.rsa_pearson_pearson = df_results_1.rsa_pearson_pearson.round(2).astype(str)
df_results_1.recall = df_results_1.recall.round(2).astype(str)
df_results_1.cosine = df_results_1.cosine.round(2).astype(str)
df_results_1.unbiased_cka = df_results_1.unbiased_cka.round(2).astype(str)
df_results_1.ssim = df_results_1.ssim.round(2).astype(str)

df_results_1 = df_results_1.rename(columns={"rsa_pearson_pearson": "rsa", "recall": "R@1", "cosine": "Cosine", "mean_rank": "Mean Rank"})
df_results_1.subject = df_results_1.subject.astype(int)
df_results_1.subject = df_results_1.subject.apply(lambda x: "Avg" if x == -1 else f"S{x}")
columns = df_results_1.subject.tolist()
df_results_1 = df_results_1.T
df_results_1.columns = columns
# Remove the first column
df_results_1 = df_results_1.iloc[1:]
df_results_1

,S1,S2,S3,S4,S5,S6,S7,S8,Avg
Mean Rank,1.03,1.12,11.25,4.69,1.77,3.09,7.6,11.67,5.28
rsa,0.6,0.61,0.47,0.51,0.6,0.52,0.5,0.43,0.53
R@1,0.99,0.98,0.75,0.79,0.87,0.83,0.72,0.62,0.82
Cosine,0.7,0.71,0.54,0.58,0.65,0.58,0.56,0.5,0.6
unbiased_cka,0.43,0.48,0.48,0.51,0.52,0.52,0.38,0.43,0.47
ssim,0.99,0.99,0.8,0.83,0.9,0.86,0.78,0.69,0.86


In [36]:
print(df_results_1.to_latex())

\begin{tabular}{llllllllll}
\toprule
 & S1 & S2 & S3 & S4 & S5 & S6 & S7 & S8 & Avg \\
\midrule
Mean Rank & 1.03 & 1.12 & 11.25 & 4.69 & 1.77 & 3.09 & 7.6 & 11.67 & 5.28 \\
rsa & 0.6 & 0.61 & 0.47 & 0.51 & 0.6 & 0.52 & 0.5 & 0.43 & 0.53 \\
R@1 & 0.99 & 0.98 & 0.75 & 0.79 & 0.87 & 0.83 & 0.72 & 0.62 & 0.82 \\
Cosine & 0.7 & 0.71 & 0.54 & 0.58 & 0.65 & 0.58 & 0.56 & 0.5 & 0.6 \\
unbiased_cka & 0.43 & 0.48 & 0.48 & 0.51 & 0.52 & 0.52 & 0.38 & 0.43 & 0.47 \\
ssim & 0.99 & 0.99 & 0.8 & 0.83 & 0.9 & 0.86 & 0.78 & 0.69 & 0.86 \\
\bottomrule
\end{tabular}



The same but with linear embeddings

In [37]:
all_results = []
embeddings_folder = Path("../scripts/linear_embeddings")
for subject in trange(1, 9):
    data = torch.load(embeddings_folder / f"ws_linear_v1_768_128_sub-{subject:02d}.pt")
    Z = data["Z"]
    df_repetitions = split_repetitions(subject=subject, shuffle_indexes=True, min_exists=1, random_state=42)
    df_repetitions = df_repetitions.query("nsd_id in @stimuli_list and shared")
    for rep_i, rep_j, inpute_rep in tqdm([(1, 2, 3), (1, 3, 2), (2, 3, 1), (2, 1, 3), (3, 1, 2), (3, 2, 1)], leave=False):
        Z_i, Z_j, nsd_ids = get_repetition_embeddings(
            rep_i=rep_i, rep_j=rep_j, other_rep=inpute_rep, Z=Z, df_repetitions=df_repetitions
        )
        results = compute_views_metrics(Z_i.cuda(), Z_j.cuda(), nsd_ids, rep_i=rep_i, rep_j=rep_j, subject=subject)
        
        # Get image based metrics
        close_ids = get_close_nsd_id(nsd_ids, Z_i.cuda(), Z_j.cuda())
        ssim_score = compute_ssim(close_ids, nsd_ids)
        results["ssim"] = ssim_score
        
        all_results.append(results)

df_results = pd.DataFrame(all_results)
df_results.to_parquet("within_subject_encoder_results-linear-encoder-3-reps.parquet", index=False)
df_results

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

,rep_i,rep_j,subject,recall,cosine,mean_rank,rsa_pearson_pearson,rsa_euclidean_pearson,unbiased_cka,ssim
0,1,2,1,0.988350,0.534760,1.116505,0.334275,0.262375,0.061062,0.988363
1,1,3,1,0.982524,0.535436,1.100971,0.333079,0.318733,0.060145,0.982871
2,2,3,1,0.982524,0.539616,1.060194,0.338603,0.296534,0.062120,0.987020
3,2,1,1,0.986408,0.534760,1.114563,0.334274,0.262375,0.061062,0.990050
4,3,1,1,0.980583,0.535436,1.106796,0.333079,0.318733,0.060145,0.984667
5,3,2,1,0.984466,0.539616,1.058252,0.338603,0.296534,0.062120,0.985845
6,1,2,2,0.970874,0.531997,1.184466,0.332195,0.375398,0.064085,0.978521
7,1,3,2,0.974757,0.532469,1.168932,0.334447,0.378511,0.062612,0.981607
8,2,3,2,0.966990,0.529888,1.273786,0.332675,0.315791,0.063204,0.979879
9,2,1,2,0.974757,0.531997,1.205825,0.332195,0.375398,0.064085,0.976374


In [38]:
send_message("Finished computing metrics for all processing steps!", attachment="within_subject_encoder_results-linear-encoder-3-reps.parquet")

In [50]:
# Table 1 - 
df_results = pd.read_parquet("within_subject_encoder_results-linear-encoder.parquet")

df_results_1 = df_results.groupby("subject").aggregate({"mean_rank": "mean", "rsa_pearson_pearson": "mean", "recall": "mean", "cosine": "mean",
                                                        "unbiased_cka": "mean", "ssim": "mean"}).reset_index()
df_results_avg = df_results_1.mean()
df_results_avg.subject = -1
# Append as last row
df_results_1 = pd.concat([df_results_1, pd.DataFrame([{"subject": "average", **df_results_avg}])], ignore_index=True)
df_results_1.subject = df_results_1.subject.astype(int).astype(str)
df_results_1.mean_rank = df_results_1.mean_rank.round(2).astype(str)
df_results_1.rsa_pearson_pearson = df_results_1.rsa_pearson_pearson.round(2).astype(str)
df_results_1.recall = df_results_1.recall.round(2).astype(str)
df_results_1.cosine = df_results_1.cosine.round(2).astype(str)
df_results_1.unbiased_cka = df_results_1.unbiased_cka.round(2).astype(str)
df_results_1.ssim = df_results_1.ssim.round(2).astype(str)

df_results_1 = df_results_1.rename(columns={"rsa_pearson_pearson": "rsa", "recall": "R@1", "cosine": "Cosine", "mean_rank": "Mean Rank"})
df_results_1.subject = df_results_1.subject.astype(int)
df_results_1.subject = df_results_1.subject.apply(lambda x: "Avg" if x == -1 else f"S{x}")
columns = df_results_1.subject.tolist()
df_results_1 = df_results_1.T
df_results_1.columns = columns
# Remove the first column
df_results_1 = df_results_1.iloc[1:]
df_results_1

,S1,S2,S3,S4,S5,S6,S7,S8,Avg
Mean Rank,1.29,1.59,31.89,17.97,5.94,12.22,29.03,40.0,17.49
rsa,0.33,0.33,0.17,0.18,0.25,0.19,0.19,0.15,0.22
R@1,0.98,0.97,0.61,0.58,0.76,0.67,0.54,0.45,0.69
Cosine,0.53,0.53,0.33,0.32,0.4,0.35,0.32,0.27,0.38
unbiased_cka,0.06,0.06,0.04,0.05,0.03,0.04,0.02,0.03,0.04
ssim,0.98,0.97,0.68,0.66,0.81,0.73,0.63,0.55,0.75


In [11]:
from fmri_mapping.io.nsd import get_subject_roi

In [39]:
all_results = []

steps_folder = Path("../scripts/steps")
templates = ["raw", "subject-{subject:02d}_betas_prec.npy", "subject-{subject:02d}_betas_res.npy", "subject-{subject:02d}_betas_rel.npy",  "subject-{subject:02d}_betas_pca.npy"]

for template in templates:
    for subject in trange(1, 9):
        if template == "raw":
            Z = get_subject_roi(subject=subject, roi=0)
        else:
            Z = np.load(steps_folder / template.format(subject=subject))
        
        df_repetitions = split_repetitions(subject=subject, shuffle_indexes=True, min_exists=1, random_state=42)
        df_repetitions = df_repetitions.query("nsd_id in @stimuli_list and shared")
        for rep_i, rep_j, inpute_rep in tqdm([(1, 2, 3), (1, 3, 2), (2, 3, 1), (2, 1, 3), (3, 1, 2), (3, 2, 1)], leave=False):
            Z_i, Z_j, nsd_ids = get_repetition_embeddings(
                rep_i=rep_i, rep_j=rep_j, other_rep=inpute_rep, Z=Z, df_repetitions=df_repetitions
            )
            Z_i, Z_j = torch.from_numpy(Z_i).float(), torch.from_numpy(Z_j).float()

            results = compute_views_metrics(Z_i, Z_j, nsd_ids, rep_i=rep_i, rep_j=rep_j, subject=subject)
            
            # Get image based metrics
            #close_ids = get_close_nsd_id(nsd_ids, Z_i, Z_j)
            #ssim_score = compute_ssim(close_ids, nsd_ids)
            #results["ssim"] = ssim_score
            if template == "raw":
                results["processing_step"] = "raw"
            else:
                results["processing_step"] = "_".join(template.split("_")[1:]).split(".")[0]
            
            
            all_results.append(results)


df_results = pd.DataFrame(all_results)
df_results.to_parquet("within_subject_results-naive-baselines-3-reps.parquet", index=False)
df_results

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

...

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

,rep_i,rep_j,subject,recall,cosine,mean_rank,rsa_pearson_pearson,rsa_euclidean_pearson,unbiased_cka,processing_step
0,1,2,1,0.081553,0.661420,116.786407,0.205989,0.058324,0.000000,raw
1,1,3,1,0.067961,0.663616,113.930099,0.324593,0.117065,0.000000,raw
2,2,3,1,0.120388,0.663037,109.054367,0.217660,0.148018,0.000000,raw
3,2,1,1,0.100971,0.661420,119.867958,0.205989,0.058324,0.000000,raw
4,3,1,1,0.104854,0.663616,118.716507,0.324593,0.117065,0.000000,raw
...,...,...,...,...,...,...,...,...,...,...
235,1,3,8,0.112621,0.277482,107.897087,0.157624,0.072263,0.121485,betas_pca
236,2,3,8,0.110680,0.288871,101.718445,0.175536,0.111135,0.136984,betas_pca
237,2,1,8,0.100971,0.277120,108.512619,0.160152,0.151145,0.133364,betas_pca
238,3,1,8,0.100971,0.277482,103.201942,0.157624,0.072263,0.121485,betas_pca


In [40]:
from dmf.alerts import send_message

send_message("Finished computing metrics for all processing steps!", attachment="within_subject_results-naive-baselines-3-reps.parquet")